# 01 · Chat Completions Basics — with a Gradio UI

**Where we are in the stack:** the bottom two layers - the **LLM** (the workload on GPU/TPU silicon) reached through the **Provider SDK** (the OpenAI/Anthropic client = the Broadcom SDK equivalent, just over HTTPS instead of PCIe).

This notebook proves three things:
1. A completion is **one forward pass** - tokens in, a probability distribution out, sampled to text.
2. The model is **stateless** - it has no memory between calls.
3. "Memory" is nothing but **re-feeding the transcript** as context.

> Everything here is *inference only*. No weights change. Ever.

In [ ]:
# Run once. Works with OpenAI, OpenRouter, or any OpenAI-compatible local server.
%pip install -q openai

In [ ]:
# --- Provider config: works with OpenAI, OpenRouter, or a local OpenAI-compatible server ---
import os
from openai import OpenAI

# Load settings from a .env file if present (falls back to existing env vars).
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    import os
    if os.path.exists(".env"):
        for _line in open(".env"):
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                os.environ.setdefault(_k.strip(), _v.strip())


# Pick ONE setup by exporting these env vars before launching Jupyter.
#
#   OpenAI:     OPENAI_BASE_URL=https://api.openai.com/v1   MODEL=gpt-4o-mini
#   OpenRouter: OPENAI_BASE_URL=https://openrouter.ai/api/v1 MODEL=openai/gpt-4o-mini
#   Local:      OPENAI_BASE_URL=http://localhost:11434/v1    MODEL=qwen2.5:7b   (Ollama)
#               (use 'qwen2.5' / 'llama3.1' etc. - a 1B model is great for chat but
#                usually too weak to drive tool-calling reliably.)

BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")
API_KEY  = os.environ.get("OPENAI_API_KEY", "set-me")   # any non-empty string for local servers
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")

# Behind a TLS-intercepting firewall/proxy, HTTPS cert verification can fail.
# Set VERIFY_SSL=false in .env to skip it: we hand the OpenAI SDK a custom
# httpx client with verification turned off. Leave it true everywhere else.
import httpx
VERIFY_SSL = os.environ.get("VERIFY_SSL", "true").strip().lower() not in ("false", "0", "no")
http_client = httpx.Client(verify=VERIFY_SSL)
if not VERIFY_SSL:
    import warnings
    warnings.filterwarnings("ignore")
    print("\u26a0\ufe0f  SSL verification DISABLED (VERIFY_SSL=false) \u2014 use only on a trusted network")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY, http_client=http_client)
print("endpoint:", BASE_URL, "| model:", MODEL)

## 1. A single completion = one forward pass

`messages` is a list of turns, each with a `role` (`system` / `user` / `assistant`) and `content`.
The call returns a distribution over next tokens, sampled into `choices[0].message.content`.

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a terse network engineer."},
        {"role": "user", "content": "In one sentence: what is a VLAN?"},
    ],
)
print(resp.choices[0].message.content)

### The two halves of inference: prefill vs decode
`prompt_tokens` = the **prefill** (your input, processed in parallel - fast).
`completion_tokens` = the **decode** (generated one token at a time - the slow, sequential part you pay latency for).

In [ ]:
u = resp.usage
print("prompt_tokens (prefill) :", u.prompt_tokens)
print("completion_tokens(decode):", u.completion_tokens)
print("total_tokens            :", u.total_tokens)

## 2. The model is stateless - watch it forget

Two **separate** calls. The second has no idea the first happened, because nothing carried over.

In [ ]:
client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": "My name is Harshith. Remember it."}],
)

second = client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": "What is my name?"}],
)
print(second.choices[0].message.content)   # it does NOT know

## 3. Memory = re-feeding the transcript (the only mechanism)

To give the model "memory", you resend the whole conversation each turn. That is *all* a chatbot does.

In [ ]:
history = [{"role": "system", "content": "You are a helpful assistant."}]

def chat(user_text):
    history.append({"role": "user", "content": user_text})
    r = client.chat.completions.create(model=MODEL, messages=history)
    reply = r.choices[0].message.content
    history.append({"role": "assistant", "content": reply})   # carry the turn forward
    return reply

print(chat("My name is Harshith."))
print(chat("What is my name?"))   # now it knows - because the transcript was resent
print("\n--- the full context the model saw on turn 2 ---")
for m in history: print(m["role"], ":", m["content"][:70])

## 4. Callback: same prompt, different answer (sampling)

`temperature=0` -> greedy (deterministic). Higher temperature -> flatter distribution -> more variety.
This is the ECMP-for-tokens point from the talk.

In [ ]:
prompt = [{"role": "user", "content": "Describe a healthy network in one short sentence."}]

print("temperature=0 (run x2):")
for _ in range(2):
    r = client.chat.completions.create(model=MODEL, messages=prompt, temperature=0)
    print("  ", r.choices[0].message.content)

print("\ntemperature=1.3 (run x2):")
for _ in range(2):
    r = client.chat.completions.create(model=MODEL, messages=prompt, temperature=1.3)
    print("  ", r.choices[0].message.content)

## Recap

- A completion = one stateless forward pass through the remote LLM.
- Memory is an illusion built by resending the transcript.
- The model only ever *talks*. It cannot *do* anything.

**Next:** to make it *act* on the world, we wrap it in a loop and give it tools. That loop is the agent - the control plane. -> `02_agent_from_scratch.ipynb`

## 5. Interactive UI — a chat that proves the "memory" trick

A tiny [Gradio](https://www.gradio.app/) chat. Every turn it **re-sends the whole
transcript** (that is the only "memory" there is, from section 3) and exposes the
**temperature** slider from section 4. Same lesson, now clickable.

In [ ]:
# Install Gradio for the interactive UI (run once).
%pip install -q gradio

In [ ]:
import gradio as gr

def chat_fn(message, history, temperature):
    # Rebuild the full transcript every call — this *is* the memory mechanism.
    messages = [{"role": "system", "content": "You are a helpful assistant."}]
    for turn in history:
        messages.append({"role": turn["role"], "content": turn["content"]})
    messages.append({"role": "user", "content": message})
    r = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=temperature,
    )
    return r.choices[0].message.content

demo = gr.ChatInterface(
    fn=chat_fn,
    additional_inputs=[gr.Slider(0.0, 1.5, value=0.7, step=0.1, label="temperature")],
    title="01 · Chat Completions",
    description="The whole transcript is re-sent every turn (that's the memory). "
                "Slide temperature up for more variety, down to 0 for deterministic.",
    examples=[["My name is Harshith. Remember it."], ["What is my name?"],
              ["In one sentence: what is a VLAN?"]],
)

demo.launch()